# Inference_exp_6_7
## Imports

In [1]:
try:
    import pylibjpeg
except:
    # Offline dependencies:
    !mkdir -p /root/.cache/torch/hub/checkpoints/

    !pip install /kaggle/input/rsna-2022-whl/{pydicom-2.3.0-py3-none-any.whl,pylibjpeg-1.4.0-py3-none-any.whl,python_gdcm-3.0.15-cp37-cp37m-manylinux_2_17_x86_64.manylinux2014_x86_64.whl}
    !pip install /kaggle/input/rsna-2022-whl/{torch-1.12.1-cp37-cp37m-manylinux1_x86_64.whl,torchvision-0.13.1-cp37-cp37m-manylinux1_x86_64.whl}

Processing /kaggle/input/rsna-2022-whl/pydicom-2.3.0-py3-none-any.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/rsna-2022-whl/pydicom-2.3.0-py3-none-any.whl'

ERROR: torch-1.12.1-cp37-cp37m-manylinux1_x86_64.whl is not a supported wheel on this platform.


In [2]:
import numpy as np
import pandas as pd
import pydicom

import cv2
import os
from tqdm import tqdm
import glob
import pickle
from albumentations import *
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models.convnext import convnext_tiny, convnext_small, convnext_base
from torchvision.models.efficientnet import efficientnet_v2_l

from torch.utils.data import Dataset, DataLoader
import torch
import sys
import time


## Config + Dataset

In [3]:
config = {'seq_len': 150,
          'feature_size': 1024,
          'lstm_size': 128,
          'target_size': 368,
          'crop_size': 320,
          'num_classes':7,
          'batch_size_image_level': 32,
          'batch_size_patient_level': 8
         }

In [4]:
#https://www.kaggle.com/code/vslaykovsky/infer-pytorch-effnetv2-single-model-lb-0-49/notebook?scriptVersionId=104434908
def load_df_test():
    df_test = pd.read_csv(f'../input/rsna-2022-cervical-spine-fracture-detection/test.csv')

    if df_test.iloc[0].row_id == '1.2.826.0.1.3680043.10197_C1':
        # test_images and test.csv are inconsistent in the dev dataset, fixing labels for the dev run.
        df_test = pd.DataFrame({
            "row_id": ['1.2.826.0.1.3680043.22327_C1', '1.2.826.0.1.3680043.25399_C1', '1.2.826.0.1.3680043.5876_C1'],
            "StudyInstanceUID": ['1.2.826.0.1.3680043.22327', '1.2.826.0.1.3680043.25399', '1.2.826.0.1.3680043.5876'],
            "prediction_type": ["C1", "C1", "patient_overall"]}
        )
    return df_test

In [5]:
test_df = load_df_test()
TEST_PATH = '../input/rsna-2022-cervical-spine-fracture-detection/test_images'
study_id_list = list(test_df.StudyInstanceUID.unique()) #uids
study_id_list

['1.2.826.0.1.3680043.6200',
 '1.2.826.0.1.3680043.27262',
 '1.2.826.0.1.3680043.12351',
 '1.2.826.0.1.3680043.1363',
 '1.2.826.0.1.3680043.4859',
 '1.2.826.0.1.3680043.18659',
 '1.2.826.0.1.3680043.17364',
 '1.2.826.0.1.3680043.849',
 '1.2.826.0.1.3680043.24045',
 '1.2.826.0.1.3680043.4744',
 '1.2.826.0.1.3680043.15773',
 '1.2.826.0.1.3680043.24946',
 '1.2.826.0.1.3680043.9290',
 '1.2.826.0.1.3680043.5482',
 '1.2.826.0.1.3680043.11090',
 '1.2.826.0.1.3680043.13810',
 '1.2.826.0.1.3680043.28252',
 '1.2.826.0.1.3680043.26052',
 '1.2.826.0.1.3680043.4131',
 '1.2.826.0.1.3680043.9996',
 '1.2.826.0.1.3680043.13814',
 '1.2.826.0.1.3680043.9121',
 '1.2.826.0.1.3680043.7824',
 '1.2.826.0.1.3680043.4561',
 '1.2.826.0.1.3680043.28657',
 '1.2.826.0.1.3680043.22438',
 '1.2.826.0.1.3680043.5565',
 '1.2.826.0.1.3680043.1381',
 '1.2.826.0.1.3680043.29548',
 '1.2.826.0.1.3680043.6739',
 '1.2.826.0.1.3680043.4438',
 '1.2.826.0.1.3680043.22019',
 '1.2.826.0.1.3680043.25093',
 '1.2.826.0.1.3680043.11718

In [6]:
selected_image_dict = {}
for uid in study_id_list:
    dicom_files = glob.glob(os.path.join(f'{TEST_PATH}/{uid}', '*.dcm'))
    # print(len(dicom_files))
    middle_slice = int(len(dicom_files)/2)
    num_left_images = num_right_images = int(0.15*len(dicom_files)) # select 15% to the left, 15% to the right
    # print(middle_slice, num_left_images)
    selected_image_dict[uid] = list(np.arange(middle_slice-num_left_images, middle_slice, 1)) +\
                               list(np.arange(middle_slice+1, middle_slice+num_right_images+1,1))

In [7]:
#print(selected_image_dict[study_id_list[0]])

## Helper functions

In [8]:
# adapted from https://www.kaggle.com/code/sparkyjunior/windowing-in-ct-scans
def window(data, WL=400, WW=1800):
    data.PhotometricInterpretation = 'YBR_FULL'
    slope     = data.RescaleSlope
    intercept = data.RescaleIntercept
    img = data.pixel_array
    img = (img*slope +intercept)
    upper, lower = WL+WW//2, WL-WW//2
    X = np.clip(img.copy(), lower, upper)
    X = X - np.min(X)
    X = X / np.max(X)
    X = (X*255.0).astype('uint8')
    return X

# https://www.kaggle.com/code/iafoss/hubmap-pytorch-fast-ai-starter
def img2tensor(img,dtype:np.dtype=np.float32):
    if img.ndim==2 : img = np.expand_dims(img,2)
    img = np.transpose(img,(2,0,1)) # since numpy array has [H,W,C] -> we want [C,H,W] for torch tensor
    return torch.from_numpy(img.astype(dtype, copy=False))

## Dataset

In [9]:
mean=np.array([0.456, 0.456, 0.456])
std=np.array([0.224, 0.224, 0.224])

class CSFImageDataset(Dataset):
    def __init__(self, uid, image_list, target_size, crop_size):
        self.uid = uid
        self.image_list  = image_list
        self.target_size = target_size
        self.crop_size   = crop_size
        
    def __len__(self):
        return len(self.image_list)
    
    def __getitem__(self, index):
        
        PATH = os.path.join(TEST_PATH, self.uid)
        data_list  = [pydicom.dcmread(f'{PATH}/{str(self.image_list[index]-1)}.dcm'), \
                      pydicom.dcmread(f'{PATH}/{str(self.image_list[index])}.dcm'), \
                      pydicom.dcmread(f'{PATH}/{str(self.image_list[index]+1)}.dcm')]
        # print(data_list)
                
        imgs  = [window(data) for data in data_list]
        
        stacked_img = np.stack(imgs, axis=-1)
        stacked_img = cv2.resize(stacked_img, (self.target_size, self.target_size))
        
#         inference_transform = Compose([CenterCrop(self.crop_size,self.crop_size)])
#         stacked_img = inference_transform(image=stacked_img)
#         # transformed image
#         X = stacked_img['image']
        X = img2tensor((stacked_img/255.0 - mean)/std)
        
        return X

In [10]:
class CSFInstanceDataset(Dataset):
    def __init__(self, feature_array_dict, study_id_list, seq_len):
        self.feature_array_dict = feature_array_dict
        self.study_id_list = study_id_list
        self.seq_len = seq_len
        
    def __len__(self):
        return len(self.study_id_list)
    
    def __getitem__(self, index):
        uid = self.study_id_list[index]
        feature_array = self.feature_array_dict[uid]
        # if a study has more slices than seq_len
        # resize features
        if len(feature_array) > self.seq_len: 
            x = cv2.resize(feature_array, (feature_array.shape[1], self.seq_len), interpolation = cv2.INTER_LINEAR)
        else:
            # a study has less slices than seq_len
            # pad with zeros   
            x = np.pad(feature_array, pad_width=[(0,self.seq_len-feature_array.shape[0]), (0,0)], constant_values=0) 
            
        X = torch.tensor(x, dtype=torch.float32) #(seq_len,1024)
        return X, uid

## Model architecture

In [11]:
class ConvNextCNN_B_Feature(nn.Module):
    def __init__(self):
        super(ConvNextCNN_B_Feature, self).__init__()
        m = convnext_base() #extract the output layer
        in_features = m.classifier[-1].in_features
        self.features = m.features
        # self.relu = nn.ReLU()
        self.avgpool = nn.AdaptiveAvgPool2d(1) 
        self.drop = nn.Dropout(p=0.5)
        self.fc = nn.Linear(in_features=in_features, out_features=7)
        
    # Progresses data across layers    
    def forward(self, x):
        out = self.features(x)
        # out = self.relu(out)
        out = self.avgpool(out)
        out = self.drop(out)
        feature = out.view(x.size(0), -1) #layer before the fc
        # shape after conv and max_pool layer [B, n_out_channel, H, W]
        # out = out.reshape(out.size(0), -1)
        
        out = self.fc(feature)
        
        return feature, out

In [12]:
class CSFNet(nn.Module):
    def __init__(self, input_len, lstm_size):
        super().__init__()
        self.lstm1 = nn.GRU(input_len, lstm_size, bidirectional=True, batch_first=True)
        self.last_linear = nn.Linear(lstm_size*2, 1) #(*4)
        # self.attention = Attention(lstm_size*2, config['seq_len'])
        
    def forward(self, x):
        h_lstm1, _ = self.lstm1(x)

        max_pool, _ = torch.max(h_lstm1, 1)
        # att_pool = self.attention(h_lstm1)
        # conc = torch.cat((max_pool, att_pool), 1)  

        logits = self.last_linear(max_pool)
        return logits

In [13]:
# Load model from `state_dict`
# Image level
lv1_model = ConvNextCNN_B_Feature()
lv1_model.load_state_dict(torch.load('../input/cnn-lstm-oct-24/model_1.pth'))
lv1_model = lv1_model.cuda()
lv1_model.eval()

# Exam level
lv2_model = CSFNet(input_len=config['feature_size'], lstm_size=config['lstm_size'])
lv2_model.load_state_dict(torch.load('../input/cnn-lstm-oct-21/run_5b/run_5b/model_lstm_3.pth'))
lv2_model = lv2_model.cuda()
lv2_model.eval()

FileNotFoundError: [Errno 2] No such file or directory: '../input/cnn-lstm-oct-24/model_1.pth'

## Inference

### Stage 1: Feature extraction

In [14]:
submission_dict = {'row_id':[], 'fractured':[]}
feature_array_dict = {}
for uid in study_id_list:
    image_list = selected_image_dict[uid]
    dataset = CSFImageDataset(uid=uid, image_list=image_list,\
                              target_size=config['target_size'], \
                              crop_size=config['crop_size'])
    generator = DataLoader(dataset, batch_size=config['batch_size_image_level'],\
                           shuffle=False, pin_memory=True, drop_last=False)
    
    preds_uid = []
    feature_array = np.zeros((len(dataset), config['feature_size']), dtype=np.float32)
    #image level prediction
    for i, images in tqdm(enumerate(generator), total=len(generator)):
        with torch.no_grad():
            start = i*config['batch_size_image_level']
            end = start+config['batch_size_image_level']
            
            if i == len(generator) - 1: # last batch
                end = len(generator.dataset)
            
            images = images.cuda()
            
            features, preds = lv1_model(images)
                
            feature_array[start:end] = np.squeeze(features.cpu().data.numpy())
            preds = preds.sigmoid().detach()
            preds_uid.append(preds)
            
    feature_array_dict[uid] = feature_array
    
    mean_preds = torch.mean(torch.cat(preds_uid), dim=0).cpu().data.numpy()
    
    
    submission_dict['row_id'].append(f'{uid}_C1')
    submission_dict['fractured'].append(mean_preds[0])
        
    submission_dict['row_id'].append(f'{uid}_C2')
    submission_dict['fractured'].append(mean_preds[1])
        
    submission_dict['row_id'].append(f'{uid}_C3')
    submission_dict['fractured'].append(mean_preds[2])
        
    submission_dict['row_id'].append(f'{uid}_C4')
    submission_dict['fractured'].append(mean_preds[3])
        
    submission_dict['row_id'].append(f'{uid}_C5')
    submission_dict['fractured'].append(mean_preds[4])
        
    submission_dict['row_id'].append(f'{uid}_C6')
    submission_dict['fractured'].append(mean_preds[5])
        
    submission_dict['row_id'].append(f'{uid}_C7')
    submission_dict['fractured'].append(mean_preds[6])
   

  0%|          | 0/3 [00:00<?, ?it/s]


RuntimeError: Unable to decompress 'JPEG Lossless, Non-Hierarchical, First-Order Prediction (Process 14 [Selection Value 1])' pixel data because all plugins are missing dependencies:
	gdcm - requires gdcm>=3.0.10
	pylibjpeg - requires pylibjpeg>=2.0 and pylibjpeg-libjpeg>=2.1

### Stage 2: Instance level prediction

In [15]:
# Exam level prediction
dataset = CSFInstanceDataset(feature_array_dict=feature_array_dict,
                             study_id_list=study_id_list,
                             seq_len=config['seq_len']
                            )
generator = DataLoader(dataset=dataset,
                       batch_size=config['batch_size_patient_level'],
                       shuffle=False,
                       pin_memory=True)

for features, list_uid in tqdm(generator, total=len(generator)):
    with torch.no_grad():
        features = features.cuda()
        preds = lv2_model(features)
        preds = np.squeeze(preds.sigmoid().cpu().data.numpy())
    
    for j in range(len(features)):# n features extracted from n patients from stage 1
        submission_dict['row_id'].append(f'{list_uid[j]}_patient_overall')
        submission_dict['fractured'].append(preds[j])

  0%|          | 0/228 [00:00<?, ?it/s]


KeyError: '1.2.826.0.1.3680043.6200'

## Submission

In [16]:
sub_df = pd.DataFrame.from_dict(submission_dict)
sub_df = sub_df.sort_values(by='row_id', axis=0, ascending=True).reset_index(drop=True)
sub_df

,row_id,fractured


In [17]:
sub_df.to_csv('submission.csv', index=False)